# ☕ Starbucks Customer Analytics: Complete Dashboard Reproduction Suite
This notebook provides the complete, systematic generation of all visualizations found in the Starbucks Analytics Dashboard. Charts are categorized exactly as they appear in the application sidebar, including all newly requested specific metrics.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Starbucks Visual Identity
S_GREEN = '#006241'
S_GOLD = '#cba258'
S_DARK = '#002B1B'
S_FOREST = '#1e3932'

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams['figure.dpi'] = 120

# Load Preprocessed Data
df = pd.read_csv('public/data/starbucks_cleaned.csv')
print(f"Loaded {len(df):,} records for analysis.")

## 📊 Overview
Strategic momentum and performance leaderboards.

In [ ]:
# 1.1 Business Momentum (Daily Order Volume)
plt.figure(figsize=(10, 5))
day_order = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
daily_counts = df.groupby('day_of_week').size().reindex(day_order)
plt.fill_between(daily_counts.index, daily_counts.values, color=S_GREEN, alpha=0.1)
plt.plot(daily_counts.index, daily_counts.values, marker='o', color=S_GREEN, linewidth=3)
plt.title('Business Momentum (Order Volume Trends)')
plt.show()

# 1.2 Product Leaderboard
plt.figure(figsize=(10, 6))
top5 = df['drink_category'].value_counts().head(5)
sns.barplot(y=top5.index, x=top5.values, color=S_GREEN)
plt.title('Product Leaderboard (Top 5 Performance)')
plt.show()

# 1.3 Loyalty Valuation
plt.figure(figsize=(10, 5))
members = df[df['is_rewards_member'] == True]
non_members = df[df['is_rewards_member'] == False]
loyalty_data = pd.DataFrame([
    {'Metric': 'Avg Spend', 'Rewards Member': members['total_spend'].mean(), 'General Customer': non_members['total_spend'].mean()},
    {'Metric': 'Avg Satisfaction', 'Rewards Member': members['customer_satisfaction'].mean(), 'General Customer': non_members['customer_satisfaction'].mean()},
    {'Metric': 'Order Ahead Rate', 'Rewards Member': (members['order_ahead'] == True).mean() * 100, 'General Customer': (non_members['order_ahead'] == True).mean() * 100}
])
loyalty_data.set_index('Metric').plot(kind='bar', color=[S_GREEN, S_GOLD], ax=plt.gca())
plt.title('Loyalty Valuation (Rewards vs. Non-Rewards)')
plt.show()

## 👥 Customers
Demographics and geographic preference maps.

In [ ]:
# 2.1 Spending by Age
plt.figure(figsize=(10, 5))
age_order = ['18-24', '25-34', '35-44', '45-54', '55+']
sns.barplot(x='customer_age_group', y='total_spend', data=df, order=age_order, color=S_GREEN)
plt.title('Spending by Age (Average Revenue per Group)')
plt.show()

# 2.2 Customer Gender
plt.figure(figsize=(7, 7))
df['customer_gender'].value_counts().plot.pie(autopct='%1.1f%%', colors=[S_GREEN, '#2E1A12'])
plt.title('Customer Gender Distribution')
plt.ylabel('')
plt.show()

# 2.3 Favorite Drinks by Group
plt.figure(figsize=(12, 6))
df.groupby(['drink_category', 'customer_gender']).size().unstack().plot(kind='bar', color=[S_GREEN, S_GOLD], ax=plt.gca())
plt.title('Favorite Drinks by Gender Group')
plt.show()

# 2.4 Orders by Region
plt.figure(figsize=(10, 5))
sns.countplot(x='region', data=df, color=S_DARK)
plt.title('Orders by Region (Geographic Reach)')
plt.show()

## 🛒 Ordering
Purchase behaviors and customization drivers.

In [ ]:
# 3.1 Types of Drinks People Buy
plt.figure(figsize=(12, 5))
sns.countplot(x='drink_category', data=df, color=S_GREEN, order=df['drink_category'].value_counts().index)
plt.title('Types of Drinks People Buy (Category Popularity)')
plt.show()

# 3.2 Extra Toppings vs. Price
plt.figure(figsize=(10, 5))
sns.barplot(x='num_customizations', y='total_spend', data=df, color=S_GOLD)
plt.title('Extra Toppings vs. Price (Revenue Uplift)')
plt.show()

## ⏰ Visit Times
Operational traffic and daily trends.

In [ ]:
# 4.1 Daily Orders
plt.figure(figsize=(10, 5))
sns.countplot(x='day_of_week', data=df, order=day_order, color=S_GREEN)
plt.title('Daily Orders (Weekly Cycle)')
plt.show()

# 4.2 Busiest Times of Day
def get_slot(time_str):
    if not isinstance(time_str, str): return 'Night'
    hour = int(time_str.split(':')[0])
    if 5 <= hour < 12: return 'Morning'
    if 12 <= hour < 17: return 'Afternoon'
    if 17 <= hour < 21: return 'Evening'
    return 'Night'
df['time_slot'] = df['order_time'].apply(get_slot)
plt.figure(figsize=(10, 5))
sns.countplot(x='time_slot', data=df, order=['Morning', 'Afternoon', 'Evening', 'Night'], color=S_FOREST)
plt.title('Busiest Times of Day')
plt.show()

## ☕ Drinks
In-depth product and digital ordering analysis.

In [ ]:
# 5.1 Popular Drink Types
plt.figure(figsize=(10, 5))
top6_drinks = df['drink_category'].value_counts().head(6)
sns.barplot(x=top6_drinks.index, y=top6_drinks.values, color=S_DARK)
plt.title('Popular Drink Types')
plt.show()

# 5.2 How People Order Ahead
plt.figure(figsize=(10, 5))
order_ahead_rate = df.groupby('order_channel')['order_ahead'].mean() * 100
sns.barplot(x=order_ahead_rate.index, y=order_ahead_rate.values, color=S_GOLD)
plt.ylabel('Order Ahead Rate (%)')
plt.title('How People Order Ahead (Channel Distribution)')
plt.show()

## 💰 Spending
Revenue drivers and demographic spending power.

In [ ]:
# 6.1 Spending by Where People Live
plt.figure(figsize=(10, 5))
sns.barplot(x='store_location_type', y='total_spend', data=df, color=S_FOREST)
plt.title('Spending by Where People Live (Location Type)')
plt.show()

# 6.2 How Extra Toppings Change the Price
plt.figure(figsize=(12, 5))
cust_spend = df.groupby('num_customizations')['total_spend'].mean()
plt.plot(cust_spend.index, cust_spend.values, marker='o', color='#D4AF37', linewidth=4)
plt.title('How Extra Toppings Change the Price (Trend Line)')
plt.show()

## 💡 Quick Facts
The high-level summary view of the entire study.

In [ ]:
# 7.1 Average Spend by Age (Quick View)
plt.figure(figsize=(10, 5))
sns.barplot(x='customer_age_group', y='total_spend', data=df, order=age_order, color=S_GREEN)
plt.title('Average Spend by Age')
plt.show()

# 7.2 How People Order (Overview)
plt.figure(figsize=(7, 7))
df['order_channel'].value_counts().plot.pie(autopct='%1.1f%%', colors=[S_GREEN, S_FOREST, S_GOLD])
plt.title('How People Order')
plt.ylabel('')
plt.show()